# 🧥 Lookzi — AI Virtual Try-On (Google Colab)

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`  
Then run each cell top-to-bottom (Shift+Enter).

| Step | What it does |
|------|--------------|
| 1 | Check GPU |
| 2 | Clone the Lookzi repo |
| 3 | Install all dependencies |
| 4 | Download model weights (~3 GB) |
| 5 | Launch the Lookzi UI with a public share link |

> Weights are downloaded once and cached. Re-runs skip the download automatically.

In [ ]:
# ── Step 1: Check GPU ─────────────────────────────────────────────────────
!nvidia-smi

import sys
print(f"Python: {sys.version.split()[0]}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("\n⚠️  No GPU detected!")
    print("   Go to Runtime → Change runtime type → T4 GPU, then re-run.")

In [ ]:
# ── Step 2: Clone Lookzi repo ─────────────────────────────────────────────
import os

REPO_DIR = '/content/Lookzi-v0.1'

if os.path.exists(REPO_DIR):
    print("Repo already present — pulling latest changes...")
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/Mohamed-Kudratov/Lookzi-v0.1.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"\nWorking directory: {os.getcwd()}")
!ls

In [ ]:
# ── Step 3: Install dependencies ──────────────────────────────────────────
# Upgrade pip first
!pip install --upgrade pip -q

# Install FASHN VTON package + all its requirements (torch, opencv, etc.)
!pip install -e . -q

# Install Lookzi web server / UI deps (not in pyproject.toml)
!pip install 'fastapi>=0.100' 'uvicorn[standard]>=0.23' 'gradio>=6.0' -q

# Verify
import importlib
importlib.invalidate_caches()

import torch
import gradio
import fashn_vton
print(f"PyTorch   : {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"Gradio    : {gradio.__version__}")
print("fashn_vton: OK")
print("\n✓ All dependencies installed")

In [ ]:
# ── Step 4: Download model weights (~3 GB, cached after first run) ────────
import os
os.chdir('/content/Lookzi-v0.1')

WEIGHTS_DIR = './weights'
MODEL_FILE  = os.path.join(WEIGHTS_DIR, 'model.safetensors')
YOLOX_FILE  = os.path.join(WEIGHTS_DIR, 'dwpose', 'yolox_l.onnx')

if os.path.exists(MODEL_FILE) and os.path.exists(YOLOX_FILE):
    print("Weights already downloaded — skipping.")
    !ls -lh {WEIGHTS_DIR}/ {WEIGHTS_DIR}/dwpose/
else:
    print("Downloading weights from HuggingFace (public repos, no token needed)...")
    print("  • fashn-ai/fashn-vton-1.5   (model.safetensors  ~2.9 GB)")
    print("  • fashn-ai/DWPose           (yolox_l.onnx, dw-ll_ucoco_384.onnx)")
    print("  • fashn-ai/fashn-human-parser (auto-cached by HuggingFace)")
    print()
    !python scripts/download_weights.py --weights-dir {WEIGHTS_DIR}

In [ ]:
# ── Step 5: Launch Lookzi ─────────────────────────────────────────────────
# A public share URL will appear below (valid for 72 hours).

import sys, os

REPO_DIR = '/content/Lookzi-v0.1'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Import the Lookzi app helpers
from app import build_ui, get_pipeline

# Load model into VRAM (once — singleton, safe to re-run)
print("Loading Lookzi model into GPU... (~30-60 seconds)")
pipe = get_pipeline()
print(f"Model ready on: {pipe.device}")

# Launch Gradio with public share link
demo = build_ui()
demo.launch(
    share=True,          # creates a public *.gradio.live URL
    debug=False,
    show_error=True,
    quiet=False,
)